In [8]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

load_dotenv()
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

In [9]:
# Initialize the state 
class BlogState(TypedDict):
    title:str
    outline:str
    content:str


#define the nodes
def create_outline(state:BlogState)->BlogState:
    response=state["title"]
    prompt =f"Generate the outline for this title :{response}"
    result=model.invoke(prompt).content

    state["outline"] = result
    return state

def create_blog(state:BlogState)->BlogState:
    response=state["outline"]
    prompt =f"Generate the blog for this outline :{response}"
    final_response = model.invoke(prompt).content
    state["content"] = final_response
    return state

#define the graph
graph = StateGraph(BlogState)

#add nodes to the graph
graph.add_node("create_outline", create_outline)
graph.add_node("create_blog", create_blog)
#add the edges to the graph
graph.add_edge(START,"create_outline")
graph.add_edge("create_outline","create_blog")
graph.add_edge("create_blog",END)

#comopile the graph
workflow = graph.compile()

print(workflow.invoke({"title":"My First Blog"}))

{'title': 'My First Blog', 'outline': 'This outline for "My First Blog" is designed to be a compelling introductory post that welcomes readers, sets expectations, and establishes the blogger\'s voice.\n\n---\n\n**Blog Post Title: My First Blog: A New Beginning!**\n\n**(Alternative Titles: Hello World! Welcome to My Corner of the Internet / The Journey Begins: My First Blog Post)**\n\n---\n\n**I. Introduction: The Grand Unveiling**\n\n*   **A. Catchy Opening Hook:**\n    *   Acknowledge the milestone ("It\'s finally happening!").\n    *   Express initial feelings (excitement, nervousness, anticipation).\n    *   A warm welcome to new readers.\n*   **B. Why "My First Blog" Matters:**\n    *   Briefly state the purpose of *this specific post* (introduction, setting the stage).\n    *   Hint at the journey leading up to this moment.\n\n**II. The "Why": My Motivation for Starting a Blog**\n\n*   **A. The Genesis of the Idea:**\n    *   How long have I considered blogging?\n    *   What insp